# MNIST Distributed Training on a Ray Cluster

This notebook demonstrates an end-to-end workflow using **kuberay-sdk** to:

1. Create a Ray cluster on Kubernetes
2. Submit a distributed MNIST training job (PyTorch + Ray Train)
3. Monitor training progress and stream logs
4. Retrieve results and clean up

**Prerequisites:**
- A Kubernetes cluster with the KubeRay operator installed
- `kubectl` configured with cluster access
- `pip install kuberay-sdk`

## 1. Setup & Authentication

The SDK uses [kube-authkit](https://pypi.org/project/kube-authkit/) for Kubernetes authentication.
Supported methods:

| Method | When to use |
|--------|------------|
| `auto` | Default — probes kubeconfig, in-cluster SA, etc. |
| `kubeconfig` | Explicit kubeconfig file path |
| `incluster` | Running inside a Kubernetes pod |
| `oidc` | OpenID Connect (Keycloak, Dex, etc.) |
| `openshift` | OpenShift OAuth |

In [ ]:
# Optional, install the sdk
#!pip install -e .. --no-build-isolation -q

In [ ]:
from kube_authkit import AuthConfig

from kuberay_sdk import KubeRayClient, SDKConfig
from kuberay_sdk.models.cluster import HeadNodeConfig
from kuberay_sdk.models.runtime_env import RuntimeEnv
from kuberay_sdk.errors import TimeoutError

In [ ]:
# Configure authentication and SDK client.
# Choose one of the auth methods below and uncomment it.

# Option 1: Auto-detect (kubeconfig / in-cluster service account)
auth = AuthConfig(
    #method="auto"
    method="openshift", 
    k8s_api_host="<K8S_API_SERVER>",
    token="<TOKEN>",
)

# Option 2: Explicit kubeconfig file
# auth = AuthConfig(method="kubeconfig", kubeconfig_path="~/.kube/config")

# Option 3: In-cluster service account (when running inside a K8s pod)
# auth = AuthConfig(method="incluster")

# Option 4: OIDC (e.g., Keycloak, Dex)
# auth = AuthConfig(
#     method="oidc",
#     oidc_issuer="https://keycloak.example.com/auth/realms/myrealm",
#     client_id="my-client",
#     use_device_flow=True,
# )

# Option 5: OpenShift OAuth
# auth = AuthConfig(method="openshift", k8s_api_host="https://api.cluster.example.com:6443")

config = SDKConfig(
    auth=auth,
    namespace="default",       # Target namespace for Ray resources
    retry_max_attempts=5,      # Retry transient K8s API errors
)
client = KubeRayClient(config=config)
print("KubeRayClient ready")

## 2. Create or Connect to a Ray Cluster

You have two options:
- **Option A:** Create a new cluster (skip if one already exists)
- **Option B:** List existing clusters and reconnect to one

The head node needs at least 8Gi to avoid OOMKill — Ray head runs GCS,
Dashboard, and other services, plus the operator mounts a Memory-backed
emptyDir (`/dev/shm`) that counts against the memory cgroup limit.

**Note (OpenShift):** On ODH/RHOAI clusters, set the annotation
`odh.ray.io/secure-trusted-network: "false"` to disable mTLS if cert-manager
is not installed.

### Option A: Create a new cluster

In [ ]:
CLUSTER_NAME = "mnist-training"

# The head node needs extra memory because:
# - The KubeRay operator mounts a Memory-backed emptyDir (/dev/shm) for Ray's
#   object store, which counts against the container's memory cgroup limit.
# - Ray head runs GCS, Dashboard, autoscaler, and other services.
cluster = client.create_cluster(
    CLUSTER_NAME,
    workers=2,
    cpus_per_worker=2,
    memory_per_worker="6Gi",
    head=HeadNodeConfig(cpus=2, memory="8Gi"),
    image="quay.io/modh/ray@sha256:595b3acd10244e33fca1ed5469dccb08df66f470df55ae196f80e56edf35ad5a",
    annotations={"odh.ray.io/secure-trusted-network": "false"},
)
print(f"Cluster '{cluster.name}' created in namespace '{cluster.namespace}'")

### Option B: List existing clusters and reconnect

Run this instead of Option A if a cluster already exists. This lets you
reconnect after a kernel restart without deleting and recreating.

In [ ]:
# List all Ray clusters in the namespace.
clusters = client.list_clusters()
for cs in clusters:
    print(f"  {cs.name:30s} {cs.state:10s} workers={cs.workers_ready}/{cs.workers_desired}  age={cs.age}")

In [ ]:
# Reconnect to an existing cluster by name.
# Change the name to match one from the list above.
CLUSTER_NAME = "mnist-training"
cluster = client.get_cluster(CLUSTER_NAME)
print(f"Connected to cluster '{cluster.name}' in namespace '{cluster.namespace}'")

In [ ]:
# Wait for the cluster to be fully ready.
print("Waiting for cluster to reach RUNNING state...")
try:
    cluster.wait_until_ready(timeout=300)
    print("Cluster is ready!")
except TimeoutError:
    print("ERROR: Cluster did not become ready within 5 minutes.")
    print("Check KubeRay operator logs and cluster events.")
    raise

In [ ]:
# Verify cluster status.
status = cluster.status()
print(f"Cluster:  {status.name}")
print(f"State:    {status.state}")
print(f"Workers:  {status.workers_ready}/{status.workers_desired}")
print(f"Ray:      {status.ray_version}")
print(f"Age:      {status.age}")

## 3. MNIST Training Script

The training script below uses **Ray Train** with **PyTorch** to distribute MNIST training
across the cluster workers. Ray Train handles:

- Data sharding across workers via `DistributedSampler`
- `DistributedDataParallel` wrapping of the model
- Metric reporting and checkpointing back to the driver

We define the script as a string and submit it to the running cluster via the
Ray Dashboard API.

In [ ]:
TRAIN_SCRIPT = '''
"""Distributed MNIST training with Ray Train + PyTorch."""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import ray.train
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer


# ----- Model -----

class MNISTNet(nn.Module):
    """Small CNN for MNIST digit classification."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # -> (B, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # -> (B, 32, 7, 7)
        x = x.view(x.size(0), -1)              # -> (B, 32*7*7)
        x = self.dropout(F.relu(self.fc1(x)))  # -> (B, 128)
        x = self.fc2(x)                        # -> (B, 10)
        return x


# ----- Training loop (runs on each worker) -----

def train_func(config):
    """Per-worker training function.

    Ray Train calls this on each worker. It handles:
    - DistributedSampler setup (via ray.train.torch.prepare_data_loader)
    - DDP model wrapping (via ray.train.torch.prepare_model)
    """
    lr = config.get("lr", 1e-3)
    epochs = config.get("epochs", 5)
    batch_size = config.get("batch_size", 64)

    # Data — torchvision downloads MNIST on the fly.
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    train_ds = datasets.MNIST("/tmp/mnist", train=True, download=True, transform=transform)
    test_ds = datasets.MNIST("/tmp/mnist", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    # Prepare for distributed training.
    # prepare_data_loader wraps with DistributedSampler.
    # prepare_model wraps with DistributedDataParallel.
    train_loader = ray.train.torch.prepare_data_loader(train_loader)
    test_loader = ray.train.torch.prepare_data_loader(test_loader)

    model = MNISTNet()
    model = ray.train.torch.prepare_model(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

        train_acc = correct / total
        avg_loss = train_loss / len(train_loader)

        # --- Evaluate ---
        model.eval()
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                _, predicted = output.max(1)
                test_total += target.size(0)
                test_correct += predicted.eq(target).sum().item()

        test_acc = test_correct / test_total

        # Report metrics back to Ray Train driver.
        ray.train.report({
            "epoch": epoch + 1,
            "train_loss": round(avg_loss, 4),
            "train_accuracy": round(train_acc, 4),
            "test_accuracy": round(test_acc, 4),
        })

        print(
            f"Epoch {epoch+1}/{epochs} — "
            f"loss: {avg_loss:.4f}, "
            f"train_acc: {train_acc:.4f}, "
            f"test_acc: {test_acc:.4f}"
        )

    print("Training complete!")


# ----- Entry point -----

if __name__ == "__main__":
    trainer = TorchTrainer(
        train_func,
        train_loop_config={
            "lr": 1e-3,
            "epochs": 5,
            "batch_size": 64,
        },
        scaling_config=ScalingConfig(
            num_workers=2,        # Matches our cluster worker count
            use_gpu=False,        # Set True if using GPU workers
        ),
    )

    result = trainer.fit()
    print(f"\\nFinal metrics: {result.metrics}")
'''

print(f"Training script: {len(TRAIN_SCRIPT)} characters")

## 4. Submit the Training Job

We submit the script to the running cluster via the Ray Dashboard API. The
script is passed inline in the entrypoint command since the Dashboard API
does not upload local directories (unlike the Ray Jobs CLI).

The `runtime_env` installs the required Python packages on the Ray workers
before the script runs.

In [ ]:
# Escape the script for shell inline execution.
import shlex

entrypoint = f"python -c {shlex.quote(TRAIN_SCRIPT)}"
print(f"Entrypoint length: {len(entrypoint)} characters")

In [ ]:
# Submit the job to the cluster.
# No working_dir needed — the script is passed inline via the entrypoint.
job = cluster.submit_job(
    entrypoint=entrypoint,
    runtime_env=RuntimeEnv(
        pip=["torch>=2.0", "torchvision", "ray[train]"],
    ),
)

print(f"Job submitted: {job.name}")

## 5. Monitor Training

Stream logs from the running job to watch training progress in real time.

In [ ]:
# Check job status.
status = job.status()
print(f"Job status: {status}")

In [ ]:
# Stream logs as training runs.
# Each line is printed as it arrives from the Ray Dashboard.
print("=" * 60)
print("TRAINING LOGS")
print("=" * 60)

for line in job.logs(stream=True, follow=True):
    print(line, end="")

In [ ]:
# Wait for the job to finish (no-op if already done from streaming).
result = job.wait(timeout=1800)
print(f"Job result: {result}")

In [ ]:
# Fetch the full logs (useful if you skipped streaming).
full_logs = job.logs()
print(full_logs)

## 6. Inspect Cluster Metrics

After training, check the cluster's resource utilization via the Ray Dashboard.

In [ ]:
metrics = cluster.metrics()
print("Cluster Metrics")
print("-" * 30)
for key, value in metrics.items():
    print(f"  {key}: {value}")

## 7. (Alternative) Standalone RayJob

Instead of creating a long-lived cluster and submitting via Dashboard, you can
use a **standalone RayJob**. This creates a disposable cluster, runs the job,
and tears everything down automatically.

This is better for batch workloads where you don't need a persistent cluster.

In [ ]:
# Uncomment to run as a standalone RayJob instead:

# standalone_job = client.create_job(
#     "mnist-standalone",
#     entrypoint=entrypoint,
#     workers=2,
#     cpus_per_worker=2,
#     memory_per_worker="6Gi",
#     head=HeadNodeConfig(cpus=2, memory="8Gi"),
#     image="rayproject/ray:2.41.0-py310",
#     runtime_env=RuntimeEnv(
#         pip=["torch>=2.0", "torchvision", "ray[train]"],
#     ),
#     shutdown_after_finish=True,  # Cluster is deleted after the job
# )
#
# result = standalone_job.wait()
# print(standalone_job.logs())

## 8. Cleanup

Delete the cluster when you're done. This removes all associated Kubernetes
resources (pods, services, config maps).

In [ ]:
cluster.delete()
print(f"Cluster '{CLUSTER_NAME}' deleted.")

In [ ]:
print("All resources cleaned up.")

## Summary

This notebook demonstrated:

| Step | SDK Method | What it does |
|------|-----------|-------------|
| Create cluster | `client.create_cluster()` | Provisions a RayCluster on K8s |
| Wait for ready | `cluster.wait_until_ready()` | Blocks until all workers are up |
| Submit job | `cluster.submit_job()` | Sends training script via Dashboard API |
| Stream logs | `job.logs(stream=True)` | Real-time log output |
| Check metrics | `cluster.metrics()` | CPU/memory utilization from Dashboard |
| Delete cluster | `cluster.delete()` | Tears down all K8s resources |

For GPU training, change `gpus_per_worker=1` in `create_cluster()` and
`use_gpu=True` in the `ScalingConfig` inside the training script.